<a href="https://colab.research.google.com/github/nathanbehailuz/fatima/blob/main/fatima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -U "huggingface_hub>=1.32.0,<2.0" transformers accelerate

In [ ]:
%pip install -q "jedi>=0.16"
%pip check

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 57.8 MB/s eta 0:00:00
No broken requirements found.


In [ ]:
import huggingface_hub
import transformers

print("huggingface_hub:", huggingface_hub.__version__)
print("transformers:", transformers.__version__)

%pip check

huggingface_hub: 1.33.0
transformers: 5.20.2
No broken requirements found.


In [ ]:
import torch

assert torch.cuda.is_available(), (
    "GPU unavailable. Select a GPU under Runtime → Change runtime type."
)

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    "GB"
)

GPU: Tesla T4
GPU memory: 14.6 GB


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import model_info

MODEL_ID = "Qwen/Qwen3.5-4B"
REVISION = model_info(MODEL_ID).sha

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=REVISION,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print("Loaded:", MODEL_ID)
print("Revision:", REVISION)

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Loaded: Qwen/Qwen3.5-4B
Revision: 851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a


In [ ]:
from transformers import TextStreamer

def ask_model(prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id,
            streamer=TextStreamer(
                tokenizer,
                skip_prompt=True,
                skip_special_tokens=True,
            ),
        )

    generated = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)

    return {
        "prompt": prompt,
        "answer": answer,
        "generated_tokens": len(generated),
        "reached_token_limit": len(generated) >= max_new_tokens,
        "generation_settings": {
            "do_sample": False,
            "repetition_penalty": 1.0,
            "max_new_tokens": max_new_tokens,
            "enable_thinking": False,
        },
    }


In [ ]:
prompt = input("Enter your prompt: ")
# Short diagnostic run; increase this limit for complete answers.
MAX_NEW_TOKENS = 1024

print("\nMODEL RESPONSE (streaming):\n")
result = ask_model(prompt, max_new_tokens=MAX_NEW_TOKENS)

print("\nGenerated tokens:", result["generated_tokens"])
print("Reached token limit:", result["reached_token_limit"])


Enter your prompt: የእፅዋት ግንድ ዋና ዋና ተግባራት ምን ምን ናቸው?

MODEL RESPONSE (streaming):

የእፅዋት ግንድ (Plant Life) በተለያዩ አካባቢዎች እና በተለያዩ ሁኔታዎች ላይ ተጽዕኖ ለሚያሳድሩ ተግባራት አሏቸው። እነዚህ ተግባራት የእፅዋትን ህይወት ለማስተካከል፣ ለአካባቢውን ለማስተካከል እና ለሌሎች ምህረቶች ለማግኘት ወሳኝ ናቸው።

የእፅዋት ግንድ ዋና ዋና ተግባራት የሚከተሉት ናቸው፡

**1. ፍጥረታትን ማፍራት (Photosynthesis)**
ይህ የእፅዋት ዋነኛው እና ወሳኝ ተግባር ነው። እፅዋት በብርሃን የሰው ልጅ (Sunlight)፣ በካርቦን ዳይኦክሳይድ (CO₂) እና በውሃ (H₂O) ይጠቀማሉ። በዚህ ሂደት ውስጥ፡-
*   ካርቦን ዳይኦክሳይድን ወደ ኦክስጅን (Oxygen) ይቀይራሉ። ይህ ኦክስጅን ለሁሉም ምህረቶች ለማለፍ ወሳኝ ነው።
*   ከላይ ያለውን ኃይል ወደ ግላይኮጂድ (Glucose) ይቀይራሉ። ይህ ግላይኮጂድ የእፅዋቱን ህይወት ለማስተካከል የሚያገለግል ኃይል (Energy) ነው።

**2. የአየርን ማጣመር (Air Purification)**
እፅዋት በተለይም በሰው ልጅ ማፍራት የሚያገለግሉት በኦክስጅን ማግኘት ነው። በተመሳሳይ፣ በሰው ልጅ ማፍራት የሚያገለግሉት በካርቦን ዳይኦክሳይድን ከአየር ውስጥ ማጣመር ነው። ይህም የአየር ጥራትን ለማሻሻል ይረዳል።

**3. የመሬትን ማጠበቅ (Soil Conservation)**
እፅዋት የመሬትን ማጠበቅ ያደርጋሉ። የእፅዋት ጭንቀት (Roots) በመሬት ውስጥ ሲገቡ፣ የመሬት ማጣትን ይከላከላሉ። ይህም የመሬትን ጥራት ይጠብቃል እና የመ

Generated tokens: 1024
Reached token limit: True


In [ ]:
import json
import transformers
from google.colab import files

record = {
    "model": MODEL_ID,
    "revision": REVISION,
    "precision": "float16",
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "generation_settings": {
        "do_sample": False,
        "repetition_penalty": 1.0,
        "max_new_tokens": 512,
    },
    **result,
}

with open("model_output.json", "w", encoding="utf-8") as f:
    json.dump(record, f, ensure_ascii=False, indent=2)

files.download("model_output.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>